# Option D — Data Acquisition and EDA Notebook

**Purpose:** Pull API data and run EDA for Option D (conflicts of interest vs evidence dissemination).
Raw data → `../../product/data/raw/` | Cleaned data → `../modified_data/`

**Sources**
1. ClinicalTrials.gov v2 API — Completed US oncology trials, primary completion 2015–2022
2. NPI Registry — investigator name → NPI resolution for payment linkage
3. CMS Open Payments — general payment records by recipient NPI (program years 2015–2022)
4. PubMed (NCBI E-utilities) — publications linked to NCT identifiers

> **Linkage chain:** Trials → overallOfficials (CT.gov) → NPI Registry → CMS Open Payments (2015–2022)
> Trials → PubMed NCT ID search → publications

**Primary outcome for modeling:** `results_posted_2yr` — did the trial post results on CT.gov within 730 days of primary completion?

In [1]:
import json
import os
import pathlib
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import requests
from dotenv import load_dotenv

DOTENV_PATH = pathlib.Path("../../../.env")
load_dotenv(dotenv_path=DOTENV_PATH)

RAW     = pathlib.Path("../../product/data/raw")
MOD     = pathlib.Path("../modified_data")
RESULTS = pathlib.Path("../results")
for d in [RAW, MOD, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

# Optional: raises PubMed rate limit from 3 to 10 req/s
NCBI_KEY = os.getenv("NCBI_API_KEY", "")

CT_URL  = "https://clinicaltrials.gov/api/v2/studies"
NPI_URL = "https://npiregistry.cms.hhs.gov/api/"
OP_BASE = "https://openpaymentsdata.cms.gov/api/1"
EUTILS  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

US_STATE_ABBR = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA",
    "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT",
    "VA","WA","WV","WI","WY","DC",
}

# Regex to strip credential/title suffixes from investigator names.
# CT.gov names arrive as "First [Middle] Last, MD" or "First Last, MD, PhD" etc.
# We want to strip everything from the first credential-like token onward.
_CRED_STRIP_RE = re.compile(
    r",?\s*(?:"
    r"M\.?D\.?|Ph\.?D\.?|D\.?O\.?|M\.?P\.?H\.?|M\.?S\.?|M\.?B\.?A\.?|"
    r"R\.?N\.?|N\.?P\.?|P\.?A\.?-?C?\.?|F\.?A\.?C\.?S\.?|F\.?R\.?C\.?S?\.?I?\.?|"
    r"B\.?S\.?N?\.?|B\.?Ch\.?|B\.?A\.?O?\.?|M\.?B\.?B\.?S\.?|"
    r"M\.?B\.?Ch\.?B\.?|D\.?P\.?H\.?|M\.?Sc\.?|D\.?Sc\.?|"
    r"Prof(?:essor)?\.?|Dr\.?|P\.?U\.?-?P\.?H\.?|"
    r"[A-Z]{2,6}\b"  # catch other all-caps abbreviations (MPH, FACS, etc.)
    r").*$",
    re.IGNORECASE,
)


def extract_date_struct(d):
    """Return ISO date string from a CT.gov date struct {date: 'YYYY-MM-DD', type: ...} or plain string."""
    if d is None:
        return None
    if isinstance(d, str):
        return d[:10]
    if isinstance(d, dict):
        v = d.get("date", "")
        return v[:10] if v else None
    return None


def parse_name_parts(full_name):
    """Extract (first, last) from a name string that may include credentials.

    CT.gov names are typically 'First [Middle] Last, MD' or 'First Last, PhD, MPH'.
    We strip credential suffixes first, then split on whitespace.
    Falls back to comma-splitting only when the pre-comma part looks like 'Last'.
    """
    name = str(full_name or "").strip()
    if not name:
        return "", ""

    # Strip credential suffixes (everything from the first credential token onward)
    clean = _CRED_STRIP_RE.sub("", name).strip().rstrip(",").strip()
    if not clean:
        # Fallback: take everything before the first comma
        clean = name.split(",")[0].strip()

    parts = clean.split()
    if len(parts) == 0:
        return "", ""
    if len(parts) == 1:
        return "", parts[0]
    # Standard CT.gov format is "First [Middle] Last"
    return parts[0], parts[-1]


sns.set_theme(style="whitegrid")
print("Raw dir    :", RAW.resolve())
print("Modified   :", MOD.resolve())
print("Results    :", RESULTS.resolve())
print("NCBI key   :", bool(NCBI_KEY))

# Quick sanity check on name parsing
_test_cases = [
    ("Ryan Sullivan, MD",             ("Ryan", "Sullivan")),
    ("James McKiernan, MD",           ("James", "McKiernan")),
    ("N. Lynn Henry, MD, PhD",        ("N.", "Henry")),
    ("Elisabeth MOYAL, professor",    ("Elisabeth", "MOYAL")),
    ("David H. Ilson, MD, PhD",       ("David", "Ilson")),
    ("Jürgen E. Gschwend, Prof. Dr.", ("Jürgen", "Gschwend")),
    ("Katherine Crew, MD",            ("Katherine", "Crew")),
    ("Diana Dickson",                 ("Diana", "Dickson")),
    ("William Robb, MB,BCh,BAO,BA,FRCSI,MD", ("William", "Robb")),
]
_ok = True
for name, expected in _test_cases:
    result = parse_name_parts(name)
    status = "OK" if result == expected else f"FAIL (got {result})"
    if status != "OK":
        _ok = False
    print(f"  {status:30s}  '{name}' → {result}")
print("Name parsing:", "all OK" if _ok else "some failures above")

Raw dir    : D:\OneDrive - University of Toronto\UofT\Year3\Winter\jsc370\JSC370-Midterm-Proj\option-d-conflicts-of-interest\product\data\raw
Modified   : D:\OneDrive - University of Toronto\UofT\Year3\Winter\jsc370\JSC370-Midterm-Proj\option-d-conflicts-of-interest\discover\modified_data
Results    : D:\OneDrive - University of Toronto\UofT\Year3\Winter\jsc370\JSC370-Midterm-Proj\option-d-conflicts-of-interest\discover\results
NCBI key   : True
  OK                              'Ryan Sullivan, MD' → ('Ryan', 'Sullivan')
  OK                              'James McKiernan, MD' → ('James', 'McKiernan')
  FAIL (got ('', 'N.'))           'N. Lynn Henry, MD, PhD' → ('', 'N.')
  FAIL (got ('', 'Eli'))          'Elisabeth MOYAL, professor' → ('', 'Eli')
  OK                              'David H. Ilson, MD, PhD' → ('David', 'Ilson')
  FAIL (got ('', 'Jü'))           'Jürgen E. Gschwend, Prof. Dr.' → ('', 'Jü')
  FAIL (got ('', 'Kat'))          'Katherine Crew, MD' → ('', 'Kat')
  OK          

---
## 1) ClinicalTrials.gov v2 — Completed US Oncology Trials (2015–2022)

Pull completed oncology trial metadata for outcome analysis. Filters applied server-side:
- Condition: `cancer OR oncology OR neoplasm OR tumor OR carcinoma OR sarcoma OR lymphoma OR leukemia`
- Status: `COMPLETED`
- Primary completion date: 2015-01-01 → 2022-12-31

Key fields extracted:
- `hasResults` — top-level boolean on the study object
- `statusModule.primaryCompletionDateStruct` / `resultsFirstPostDateStruct`
- `sponsorCollaboratorsModule.leadSponsor.class`
- `designModule.phases` / `enrollmentInfo`
- `contactsLocationsModule.overallOfficials` — name, role, affiliation
- `conditionsModule.conditions` — for condition-group classification

In [2]:
version_payload = requests.get("https://clinicaltrials.gov/api/v2/version", timeout=30).json()
(RAW / "ct_d_version.json").write_text(json.dumps(version_payload, indent=2), encoding="utf-8")
print("Saved:", RAW / "ct_d_version.json")
print("Version payload keys:", list(version_payload.keys()))
print(version_payload)

Saved: ..\..\product\data\raw\ct_d_version.json
Version payload keys: ['apiVersion', 'dataTimestamp']
{'apiVersion': '2.0.5', 'dataTimestamp': '2026-03-04T11:00:05'}


In [3]:
CT_FIELDS = ",".join([
    "protocolSection.identificationModule.nctId",
    "protocolSection.identificationModule.officialTitle",
    "protocolSection.identificationModule.briefTitle",
    "protocolSection.statusModule.overallStatus",
    "protocolSection.statusModule.primaryCompletionDateStruct",
    "protocolSection.statusModule.completionDateStruct",
    "protocolSection.statusModule.resultsFirstPostDateStruct",
    "protocolSection.statusModule.studyFirstPostDateStruct",
    "protocolSection.sponsorCollaboratorsModule.leadSponsor",
    "protocolSection.designModule.phases",
    "protocolSection.designModule.enrollmentInfo",
    "protocolSection.contactsLocationsModule.overallOfficials",
    "protocolSection.conditionsModule.conditions",
])

base_params = {
    "query.cond":      "cancer OR oncology OR neoplasm OR tumor OR carcinoma OR sarcoma OR lymphoma OR leukemia",
    "filter.overallStatus": "COMPLETED",
    "filter.advanced": "AREA[PrimaryCompletionDate]RANGE[2015-01-01,2022-12-31]",
    "pageSize":        1000,
    "countTotal":      "true",
    "fields":          CT_FIELDS,
}

page_token = None
page_num   = 0
total      = 0

while True:
    params = dict(base_params)
    if page_token:
        params["pageToken"] = page_token

    resp = requests.get(CT_URL, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()

    studies = payload.get("studies", [])
    if not studies:
        print("No studies returned; stopping.")
        break

    out = RAW / f"ct_d_studies_page{page_num}.json"
    out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    total += len(studies)
    print(f"Saved page {page_num}: {len(studies)} studies -> {out.name}")

    page_token = payload.get("nextPageToken")
    if not page_token:
        break

    page_num += 1
    time.sleep(0.2)

print(f"\nDone. Total oncology completed trials collected: {total}")

Saved page 0: 1000 studies -> ct_d_studies_page0.json
Saved page 1: 1000 studies -> ct_d_studies_page1.json
Saved page 2: 1000 studies -> ct_d_studies_page2.json
Saved page 3: 1000 studies -> ct_d_studies_page3.json
Saved page 4: 1000 studies -> ct_d_studies_page4.json
Saved page 5: 1000 studies -> ct_d_studies_page5.json
Saved page 6: 1000 studies -> ct_d_studies_page6.json
Saved page 7: 1000 studies -> ct_d_studies_page7.json
Saved page 8: 1000 studies -> ct_d_studies_page8.json
Saved page 9: 1000 studies -> ct_d_studies_page9.json
Saved page 10: 1000 studies -> ct_d_studies_page10.json
Saved page 11: 1000 studies -> ct_d_studies_page11.json
Saved page 12: 1000 studies -> ct_d_studies_page12.json
Saved page 13: 1000 studies -> ct_d_studies_page13.json
Saved page 14: 1000 studies -> ct_d_studies_page14.json
Saved page 15: 1000 studies -> ct_d_studies_page15.json
Saved page 16: 1000 studies -> ct_d_studies_page16.json
Saved page 17: 1000 studies -> ct_d_studies_page17.json
Saved page 1

In [4]:
page_files = sorted(RAW.glob("ct_d_studies_page*.json"))
if not page_files:
    raise FileNotFoundError("No CT page files found. Run the fetch cell first.")

# Keywords for condition classification
_HEMATOLOGIC = {"leukemia", "lymphoma", "myeloma", "myeloid", "lymphoid", "hodgkin"}
_SOLID       = {"carcinoma", "cancer", "tumor", "tumour", "sarcoma", "adenocarcinoma",
                "melanoma", "glioma", "glioblastoma", "mesothelioma", "neuroblastoma"}

def classify_condition(conditions_list):
    text = " ".join(str(c).lower() for c in (conditions_list or []))
    if any(k in text for k in _HEMATOLOGIC):
        return "hematologic"
    if any(k in text for k in _SOLID):
        return "solid_tumor"
    return "other"

trial_rows = []
inv_rows   = []

for page in page_files:
    payload = json.loads(page.read_text(encoding="utf-8"))
    for study in payload.get("studies", []):
        protocol   = study.get("protocolSection", {})
        ident      = protocol.get("identificationModule", {})
        status     = protocol.get("statusModule", {})
        design     = protocol.get("designModule", {})
        sponsor    = protocol.get("sponsorCollaboratorsModule", {})
        contacts   = protocol.get("contactsLocationsModule", {})
        conditions = protocol.get("conditionsModule", {}).get("conditions", [])

        nct_id = ident.get("nctId")
        if not nct_id:
            continue

        # hasResults is a top-level field on the study object
        has_results   = bool(study.get("hasResults", False))
        phases        = design.get("phases", [])
        enrollment    = design.get("enrollmentInfo", {})
        lead          = sponsor.get("leadSponsor", {}) if isinstance(sponsor, dict) else {}
        sponsor_class = str(lead.get("class", "")).upper()

        trial_rows.append({
            "nct_id":                  nct_id,
            "title":                   ident.get("officialTitle") or ident.get("briefTitle", ""),
            "overall_status":          status.get("overallStatus"),
            "completion_date":         extract_date_struct(status.get("completionDateStruct")),
            "primary_completion_date": extract_date_struct(status.get("primaryCompletionDateStruct")),
            "study_first_post_date":   extract_date_struct(status.get("studyFirstPostDateStruct")),
            "results_first_post_date": extract_date_struct(status.get("resultsFirstPostDateStruct")),
            "has_results":             has_results,
            "phase":                   ", ".join(phases) if isinstance(phases, list) else str(phases),
            "enrollment":              enrollment.get("count") if isinstance(enrollment, dict) else None,
            "lead_sponsor_class":      sponsor_class,
            "is_industry":             1 if sponsor_class == "INDUSTRY" else 0,
            "conditions":              "; ".join(conditions) if conditions else "",
            "condition_group":         classify_condition(conditions),
        })

        officials = contacts.get("overallOfficials", []) if isinstance(contacts, dict) else []
        for off in (officials or []):
            if not isinstance(off, dict):
                continue
            name = str(off.get("name", "")).strip()
            if name:
                inv_rows.append({
                    "nct_id":            nct_id,
                    "investigator_name": name,
                    "role":              off.get("role", ""),
                    "affiliation":       off.get("affiliation", ""),
                })

df_trials = pd.DataFrame(trial_rows).drop_duplicates(subset=["nct_id"]).copy()
df_inv = (
    pd.DataFrame(inv_rows)
    if inv_rows
    else pd.DataFrame(columns=["nct_id", "investigator_name", "role", "affiliation"])
)

for col in ["completion_date", "primary_completion_date", "study_first_post_date", "results_first_post_date"]:
    df_trials[col] = pd.to_datetime(df_trials[col], errors="coerce")

df_trials["enrollment"] = pd.to_numeric(df_trials["enrollment"], errors="coerce")

# Primary outcome: results posted within 730 days of primary completion
df_trials["days_to_results"] = (
    (df_trials["results_first_post_date"] - df_trials["primary_completion_date"]).dt.days
)
df_trials["results_posted_2yr"] = (
    df_trials["days_to_results"].notna() & (df_trials["days_to_results"] <= 730)
).astype(int)

# Completion year for Open Payments window matching
df_trials["completion_year"] = df_trials["primary_completion_date"].dt.year

# Log-enrollment feature (used in modeling)
df_trials["log_enrollment"] = np.log1p(df_trials["enrollment"].fillna(0))

df_trials.to_csv(MOD / "ct_d_trials_flat.csv", index=False)
df_inv.to_csv(MOD / "ct_d_investigators.csv", index=False)

print("Trials              :", len(df_trials))
print("Investigator rows   :", len(df_inv))
print("Has results         :", df_trials["has_results"].sum(), "/", len(df_trials))
print("Results posted 2yr  :", df_trials["results_posted_2yr"].sum(), "/", df_trials["has_results"].sum(), "(of those with results)")
print("Industry sponsor    :", df_trials["is_industry"].sum())
print("Unique investigators:", df_inv["investigator_name"].nunique())
print("\nCondition groups:")
print(df_trials["condition_group"].value_counts())
df_trials[["nct_id", "phase", "enrollment", "has_results", "results_posted_2yr",
           "days_to_results", "lead_sponsor_class", "condition_group"]].head(5)

Trials              : 23075
Investigator rows   : 22352
Has results         : 0 / 23075
Results posted 2yr  : 781 / 0 (of those with results)
Industry sponsor    : 5438
Unique investigators: 17223

Condition groups:
condition_group
solid_tumor    15195
other           5012
hematologic     2868
Name: count, dtype: int64


,nct_id,phase,enrollment,has_results,results_posted_2yr,days_to_results,lead_sponsor_class,condition_group
0,NCT01872221,NA,21.0,False,0,NaN,OTHER,solid_tumor
1,NCT02908178,,28291.0,False,0,NaN,OTHER,solid_tumor
2,NCT01389440,PHASE2,24.0,False,0,NaN,OTHER,solid_tumor
3,NCT01784640,PHASE1,13.0,False,0,NaN,OTHER,solid_tumor
4,NCT00538668,PHASE1,55.0,False,0,NaN,OTHER,solid_tumor


---
## 2) NPI Registry — Investigator Name → NPI Resolution

Look up each unique investigator name from ClinicalTrials.gov against the NPI Registry.
Results are cached per investigator to avoid redundant API calls.

**Note:** If you previously ran this cell with the old (buggy) `parse_name_parts` that sent
credentials as first names, delete the cache directory first:
```bash
rm -rf ../../product/data/raw/npi_investigators/
```
Then re-run this cell.

**Expected match rates for oncology CT.gov names:**
- Many investigators are non-US → legitimately 0 hits
- US investigators with common names may return `multiple` hits
- US investigators with unique names → `exact_unique`

**Match confidence strata:**
- `exact_unique` — exactly 1 NPI returned (most reliable)
- `multiple`     — ≥ 2 NPIs returned (first result retained; flagged as ambiguous)
- `none`         — 0 results (non-US investigators or name not found)

In [6]:
NPI_INV_DIR = RAW / "npi_investigators"
NPI_INV_DIR.mkdir(exist_ok=True)

unique_names = df_inv["investigator_name"].dropna().unique().tolist()
print(f"Unique investigator names to look up: {len(unique_names)}")

npi_results = {}
for i, name in enumerate(unique_names):
    safe  = re.sub(r"[^\w-]", "_", name)[:80]
    cache = NPI_INV_DIR / f"npi_{safe}.json"

    if cache.exists():
        data = json.loads(cache.read_text(encoding="utf-8"))
        if i % 200 == 0:
            print(f"  [{i}/{len(unique_names)}] cached: {name[:50]}", flush=True)
    else:
        first, last = parse_name_parts(name)
        if not last:
            data = {"results": [], "result_count": 0}
        else:
            r = requests.get(NPI_URL, params={
                "version":          "2.1",
                "enumeration_type": "NPI-1",
                "first_name":       first,
                "last_name":        last,
                "limit":            5,
            }, timeout=30)
            data = r.json() if r.status_code == 200 else {"results": [], "result_count": 0}
        cache.write_text(json.dumps(data), encoding="utf-8")
        if i % 100 == 0:
            print(f"  [{i}/{len(unique_names)}] fetched: {name[:50]} -> {data.get('result_count', 0)} hits", flush=True)
        time.sleep(0.1)

    results = data.get("results") or []
    count   = len(results)
    if count == 0:
        npi_results[name] = {"npi": None,                       "match_type": "none",         "match_count": 0}
    elif count == 1:
        npi_results[name] = {"npi": results[0].get("number"),   "match_type": "exact_unique", "match_count": 1}
    else:
        npi_results[name] = {"npi": results[0].get("number"),   "match_type": "multiple",     "match_count": count}

df_npi = pd.DataFrame([
    {"investigator_name": name, **vals} for name, vals in npi_results.items()
])
df_npi.to_csv(MOD / "ct_d_investigator_npi_lookup.csv", index=False)

print("\nNPI match summary:")
print(df_npi["match_type"].value_counts())

matched_npis = set(df_npi.loc[df_npi["npi"].notna(), "npi"].astype(str).tolist())
print(f"\nNPIs available for Open Payments query: {len(matched_npis)}")

Unique investigator names to look up: 17223
  [0/17223] cached: Elisabeth MOYAL, professor
  [200/17223] cached: James McKiernan, MD
  [400/17223] cached: Ryan Sullivan, MD
  [600/17223] cached: Mohammed A Ghanem, MD
  [800/17223] cached: Emmanuel Coron, Pr
  [1000/17223] cached: Paul PEREZ, MD
  [1200/17223] cached: N. Lynn Henry, MD, PhD
  [1400/17223] cached: William Robb, MB,BCh,BAO,BA,FRCSI,MD
  [1600/17223] cached: Tong weidong, Professor
  [1800/17223] cached: Sarah A. Kelleher, PhD
  [2000/17223] cached: Radka Stoyanova, PhD
  [2200/17223] cached: XIAOLIANG HAN, Ph.D.
  [2400/17223] cached: David H. Ilson, MD, PhD
  [2600/17223] cached: Michael Cleary, MD
  [2800/17223] cached: Guy H Montgomery, PhD
  [3000/17223] cached: Diana Dickson
  [3200/17223] cached: Adrian Bak, MD
  [3400/17223] cached: Esther Troost, PhD
  [3600/17223] cached: Toru Sugiyama, MD
  [3800/17223] cached: Maurizio Muscaritoli, Prof.
  [4000/17223] cached: Amr Qudeimat, MD
  [4200/17223] cached: Emmanuel DE

---
## 3) CMS Open Payments — Industry Payments by NPI (2015–2022)

Query the CMS Open Payments DKAN API (`openpaymentsdata.cms.gov`) for general-payment
records where the recipient NPI is in our matched investigator set, covering **all program
years 2015–2022** to align with the trial completion window.

**Step 3a** — Discover all General Payments datasets (one per program year) via DKAN metastore.

**Step 3b** — For each year's dataset, batch-query matched NPIs in groups of 150.

**Step 3c** — Aggregate to `npi × program_year` totals; used in the linkage step to
apply a ±2 year payment window relative to each trial's `completion_year`.

Columns pulled:
`Covered_Recipient_NPI`, `Record_ID`, `Total_Amount_of_Payment_USDollars`,
`Date_of_Payment`, `Program_Year`, `Form_of_Payment_or_Transfer_of_Value`,
`Nature_of_Payment_or_Transfer_of_Value`,
`Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name`

In [7]:
catalog_url = f"{OP_BASE}/metastore/schemas/dataset/items"
cat_resp = requests.get(catalog_url, timeout=60)
cat_resp.raise_for_status()
catalog = cat_resp.json()

(RAW / "op_catalog.json").write_text(json.dumps(catalog, indent=2), encoding="utf-8")
print(f"Catalog items: {len(catalog)}")

TARGET_YEARS = set(range(2015, 2023))  # 2015–2022 inclusive

# Filter to General Payments datasets for target years (exclude Research Payments)
import re as _re

def _extract_year(title):
    m = _re.search(r"\b(201[5-9]|202[0-2])\b", str(title))
    return int(m.group()) if m else None

gp_datasets = []
for d in catalog:
    if not isinstance(d, dict):
        continue
    title = str(d.get("title", "")).lower()
    if "general" not in title or "payment" not in title or "research" in title:
        continue
    year = _extract_year(d.get("title", ""))
    if year in TARGET_YEARS:
        d["_program_year"] = year
        gp_datasets.append(d)

gp_datasets.sort(key=lambda d: d["_program_year"])

print(f"\nGeneral Payment datasets found for 2015–2022: {len(gp_datasets)}")
for gp in gp_datasets:
    print(f"  [{gp['_program_year']}] [{gp.get('identifier')}] {gp.get('title')}")

Catalog items: 74

General Payment datasets found for 2015–2022: 5
  [2018] [f003634c-c103-568f-876c-73017fa83be0] 2018 General Payment Data
  [2019] [4e54dd6c-30f8-4f86-86a7-3c109a89528e] 2019 General Payment Data
  [2020] [a08c4b30-5cf3-4948-ad40-36f404619019] 2020 General Payment Data
  [2021] [0380bbeb-aea1-58b6-b708-829f92a48202] 2021 General Payment Data
  [2022] [df01c2f8-dc1f-4e79-96cb-8208beaf143c] 2022 General Payment Data


In [8]:
if not gp_datasets:
    raise RuntimeError(
        "No General Payments datasets found for 2015–2022. "
        "Check OP_BASE URL or update TARGET_YEARS regex."
    )

OP_RAW_DIR = RAW / "op_payments"
OP_RAW_DIR.mkdir(exist_ok=True)

npi_list = sorted(matched_npis)
BATCH    = 150  # stay well under URL length limit

all_op_records = []  # accumulates across all years

for gp_dataset in gp_datasets:
    program_year  = gp_dataset["_program_year"]
    distributions = gp_dataset.get("distribution", [])

    # Extract DKAN datastore resource ID
    resource_id = None
    for dist in distributions:
        rid = dist.get("identifier")
        if rid:
            resource_id = rid
            break

    if not resource_id:
        print(f"  [{program_year}] No resource ID found — skipping.")
        continue

    print(f"\n[{program_year}] Dataset: {gp_dataset.get('title')} | resource: {resource_id}")
    year_records = []

    for batch_start in range(0, max(len(npi_list), 1), BATCH):
        batch     = npi_list[batch_start : batch_start + BATCH]
        cache_key = f"op_{program_year}_npi_batch_{batch_start:05d}.json"
        cache     = OP_RAW_DIR / cache_key

        if cache.exists():
            records = json.loads(cache.read_text(encoding="utf-8"))
            if batch_start % 1500 == 0:
                print(f"  [{program_year}] Batch {batch_start:5d}: cached ({len(records)} rows)", flush=True)
        else:
            if not batch:
                records = []
            else:
                npi_str = "','".join(batch)
                sql = f"[SELECT * FROM {resource_id} WHERE Covered_Recipient_NPI IN ('{npi_str}')]"
                r = requests.get(
                    f"{OP_BASE}/datastore/sql",
                    params={"query": sql, "show_db_columns": "true"},
                    timeout=120,
                )
                if r.status_code != 200:
                    print(f"  [{program_year}] Batch {batch_start}: HTTP {r.status_code} — {r.text[:120]}", flush=True)
                    records = []
                else:
                    payload = r.json()
                    records = payload if isinstance(payload, list) else payload.get("results", [])
            cache.write_text(json.dumps(records), encoding="utf-8")
            print(f"  [{program_year}] Batch {batch_start:5d}: fetched {len(records)} rows", flush=True)
            time.sleep(0.3)

        # Tag each record with its program year in case the column is missing
        for rec in records:
            if isinstance(rec, dict):
                rec.setdefault("_program_year", program_year)
        year_records.extend(records)

    print(f"  [{program_year}] Total rows: {len(year_records)}")
    all_op_records.extend(year_records)

print(f"\nAll years combined: {len(all_op_records)} rows")
(RAW / "op_general_payments_matched.json").write_text(
    json.dumps(all_op_records, indent=2), encoding="utf-8"
)
print("Saved: op_general_payments_matched.json")

  [2018] No resource ID found — skipping.
  [2019] No resource ID found — skipping.
  [2020] No resource ID found — skipping.
  [2021] No resource ID found — skipping.
  [2022] No resource ID found — skipping.

All years combined: 0 rows
Saved: op_general_payments_matched.json


In [9]:
op_records = all_op_records  # alias for downstream cells

if not op_records:
    print("No Open Payments records retrieved — check NPI list and resource ID.")
    df_payments = pd.DataFrame(columns=[
        "npi", "record_id", "payment_amount", "payment_date",
        "program_year", "payment_form", "payment_nature", "company_name",
    ])
    df_npi_year = pd.DataFrame(columns=["npi", "program_year", "year_payment", "year_count"])
    df_npi_agg  = pd.DataFrame(columns=["npi", "total_payment", "payment_count"])
else:
    df_raw_op = pd.DataFrame(op_records)
    df_raw_op.columns = [c.lower() for c in df_raw_op.columns]

    # Flexible column resolution — names may vary slightly across program years
    col_map = {
        "npi":            next((c for c in df_raw_op.columns if "recipient_npi" in c), None),
        "record_id":      next((c for c in df_raw_op.columns if "record_id" in c), None),
        "payment_amount": next((c for c in df_raw_op.columns if "amount_of_payment" in c or "total_amount" in c), None),
        "payment_date":   next((c for c in df_raw_op.columns if "date_of_payment" in c), None),
        "payment_form":   next((c for c in df_raw_op.columns if "form_of_payment" in c), None),
        "payment_nature": next((c for c in df_raw_op.columns if "nature_of_payment" in c), None),
        "company_name":   next((c for c in df_raw_op.columns if "making_payment_name" in c), None),
        "program_year":   next((c for c in df_raw_op.columns if c == "program_year"), "_program_year"),
    }
    print("Column mapping:", {k: v for k, v in col_map.items() if v})

    df_payments = df_raw_op.rename(columns={v: k for k, v in col_map.items() if v and v != k}).copy()

    if "payment_amount" in df_payments.columns:
        df_payments["payment_amount"] = pd.to_numeric(df_payments["payment_amount"], errors="coerce")
    if "payment_date" in df_payments.columns:
        df_payments["payment_date"] = pd.to_datetime(df_payments["payment_date"], errors="coerce")

    # Resolve program_year: prefer the API column, fall back to _program_year tag
    if "program_year" in df_payments.columns:
        df_payments["program_year"] = pd.to_numeric(df_payments["program_year"], errors="coerce")
    if "_program_year" in df_payments.columns:
        df_payments["program_year"] = df_payments["program_year"].fillna(df_payments["_program_year"])

    df_payments["program_year"] = df_payments["program_year"].astype("Int64")

    # --- NPI × program_year aggregation (used for ±2yr windowed payment in linkage step) ---
    group_cols = ["npi", "program_year"]
    df_npi_year = (
        df_payments.dropna(subset=["npi"])
        .groupby(group_cols, as_index=False)["payment_amount"]
        .agg(year_payment="sum", year_count="count")
    )

    # --- Overall NPI aggregation (all years combined, for summary plots) ---
    df_npi_agg = (
        df_payments.dropna(subset=["npi"])
        .groupby("npi", as_index=False)["payment_amount"]
        .agg(total_payment="sum", payment_count="count")
    )
    if "payment_nature" in df_payments.columns:
        top_nature = (
            df_payments.dropna(subset=["npi"])
            .groupby("npi")["payment_nature"]
            .agg(lambda x: x.mode().iloc[0] if len(x) else pd.NA)
            .reset_index()
            .rename(columns={"payment_nature": "top_payment_nature"})
        )
        df_npi_agg = df_npi_agg.merge(top_nature, on="npi", how="left")

keep_cols = [c for c in ["npi", "record_id", "payment_amount", "payment_date",
                          "program_year", "payment_form", "payment_nature", "company_name"]
             if c in df_payments.columns]
df_payments = df_payments[keep_cols].copy()

df_payments.to_csv(MOD / "op_d_payments_flat.csv", index=False)
df_npi_year.to_csv(MOD / "op_d_payments_by_npi_year.csv", index=False)
df_npi_agg.to_csv(MOD / "op_d_payments_by_npi.csv", index=False)

print(f"Payment rows           : {len(df_payments)}")
if "payment_amount" in df_payments.columns and len(df_payments):
    print(f"Amount range           : ${df_payments['payment_amount'].min():,.2f} – ${df_payments['payment_amount'].max():,.2f}")
    print(f"Total paid (all years) : ${df_payments['payment_amount'].sum():,.2f}")
print(f"NPI × year rows        : {len(df_npi_year)}")
print(f"NPIs with payment data : {len(df_npi_agg)}")
if "program_year" in df_payments.columns:
    print("\nRows per program year:")
    print(df_payments["program_year"].value_counts().sort_index())
df_npi_year.head()

No Open Payments records retrieved — check NPI list and resource ID.
Payment rows           : 0
NPI × year rows        : 0
NPIs with payment data : 0

Rows per program year:
Series([], Name: count, dtype: int64)


,npi,program_year,year_payment,year_count


---
## 4) PubMed (NCBI E-utilities) — Publications Linked to NCT IDs

Search PubMed for each NCT ID to find associated publications.

- **`esearch`** — finds PMIDs matching `NCT{id}[Secondary Source ID]` (precise registry linkage per PLAN)
- **`esummary`** — retrieves pub date, journal, and title for found PMIDs

Responses are cached per NCT ID under `product/data/raw/pubmed_nct/`.
Rate delay: 0.12 s with `NCBI_API_KEY` set, 0.35 s without (3 req/s limit).

In [11]:
PUBMED_RAW_DIR = RAW / "pubmed_nct"
PUBMED_RAW_DIR.mkdir(exist_ok=True)

nct_ids    = df_trials["nct_id"].dropna().unique().tolist()
rate_delay = 0.12 if NCBI_KEY else 0.35

pub_rows = []
print(f"Searching PubMed for {len(nct_ids)} NCT IDs (rate delay {rate_delay}s per call) ...")

for i, nct_id in enumerate(nct_ids):
    cache = PUBMED_RAW_DIR / f"pm_{nct_id}.json"

    if cache.exists():
        saved     = json.loads(cache.read_text(encoding="utf-8"))
        pmids     = saved.get("pmids", [])
        summaries = saved.get("summaries", {})
    else:
        # esearch: use [Secondary Source ID] for precise NCT linkage (PLAN spec)
        search_params = {
            "db":      "pubmed",
            "term":    f"{nct_id}[Secondary Source ID]",
            "retmode": "json",
            "retmax":  10,
        }
        if NCBI_KEY:
            search_params["api_key"] = NCBI_KEY

        sr    = requests.get(f"{EUTILS}esearch.fcgi", params=search_params, timeout=30)
        pmids = []
        if sr.status_code == 200:
            pmids = sr.json().get("esearchresult", {}).get("idlist", [])

        # esummary for any found PMIDs
        summaries = {}
        if pmids:
            time.sleep(rate_delay)
            sum_params = {
                "db":      "pubmed",
                "id":      ",".join(pmids),
                "retmode": "json",
            }
            if NCBI_KEY:
                sum_params["api_key"] = NCBI_KEY
            sumr = requests.get(f"{EUTILS}esummary.fcgi", params=sum_params, timeout=30)
            if sumr.status_code == 200:
                summaries = sumr.json().get("result", {})
                summaries.pop("uids", None)

        cache.write_text(
            json.dumps({"pmids": pmids, "summaries": summaries}), encoding="utf-8"
        )
        time.sleep(rate_delay)

    for pmid in pmids:
        s = summaries.get(str(pmid), {})
        pub_rows.append({
            "nct_id":   nct_id,
            "pmid":     pmid,
            "pub_date": s.get("pubdate", ""),
            "journal":  s.get("fulljournalname", s.get("source", "")),
            "title":    s.get("title", ""),
        })

    if i % 500 == 0:
        print(f"  [{i:5d}/{len(nct_ids)}] {nct_id}: {len(pmids)} PMID(s)", flush=True)

df_pubs = (
    pd.DataFrame(pub_rows)
    if pub_rows
    else pd.DataFrame(columns=["nct_id", "pmid", "pub_date", "journal", "title"])
)
df_pubs["pub_date"] = pd.to_datetime(df_pubs["pub_date"], errors="coerce")
df_pubs.to_csv(MOD / "pubmed_d_publications.csv", index=False)

print(f"\nTotal publication rows  : {len(df_pubs)}")
print(f"Trials with >= 1 PMID  : {df_pubs['nct_id'].nunique()}")
print(f"Coverage (% of trials) : {df_pubs['nct_id'].nunique() / max(len(nct_ids), 1) * 100:.1f}%")
df_pubs.head()

Searching PubMed for 23075 NCT IDs (rate delay 0.12s per call) ...
  [    0/23075] NCT01872221: 1 PMID(s)
  [  500/23075] NCT02764866: 1 PMID(s)
  [ 1000/23075] NCT03854799: 0 PMID(s)
  [ 1500/23075] NCT04694820: 0 PMID(s)
  [ 2000/23075] NCT05173545: 0 PMID(s)
  [ 2500/23075] NCT04050085: 0 PMID(s)
  [ 3000/23075] NCT04372862: 0 PMID(s)
  [ 3500/23075] NCT04186754: 1 PMID(s)
  [ 4000/23075] NCT02042105: 0 PMID(s)
  [ 4500/23075] NCT03816774: 0 PMID(s)
  [ 5000/23075] NCT04782713: 1 PMID(s)


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))

---
## 5) Record Linkage — Build Master Analysis Table

Combine all four sources into one trial-level analysis table:

| Source | Join key | Fields added |
|---|---|---|
| CT.gov trials | `nct_id` | results status, dates, sponsor class, enrollment, condition_group |
| CT.gov investigators → NPI Registry | `investigator_name` | `npi`, `match_type` |
| Open Payments (±2yr window) | `npi` + `program_year` | `trial_total_payment` (windowed) |
| PubMed | `nct_id` | `pub_count`, `first_pub_date` |

**Payment window:** For each trial, sum investigator payments from Open Payments where
`abs(program_year − completion_year) ≤ 2`.

Derived columns for modeling:
- `results_posted_2yr` — primary outcome (already on `df_trials`)
- `log1p_payment = log(1 + trial_total_payment)` — continuous payment predictor
- `log_enrollment` — already on `df_trials`
- `payment_tier` — quartile label (Q1_low … Q4_high) or `no_payment_data`
- `linkage_confidence` — `all_exact` / `partial` / `none`

In [ ]:
# Step 1: attach NPI match info to each investigator row
df_inv_npi = df_inv.merge(
    df_npi[["investigator_name", "npi", "match_type", "match_count"]],
    on="investigator_name",
    how="left",
)

# Step 2: compute windowed payments per investigator using NPI × year table
# Join on NPI, then filter by program_year relative to each trial's completion_year
df_inv_npi_ct = df_inv_npi.merge(
    df_trials[["nct_id", "completion_year"]],
    on="nct_id",
    how="left",
)

if not df_npi_year.empty and "npi" in df_npi_year.columns:
    df_inv_npi_pay = df_inv_npi_ct.merge(
        df_npi_year[["npi", "program_year", "year_payment"]],
        on="npi",
        how="left",
    )
    df_inv_npi_pay["program_year"] = pd.to_numeric(df_inv_npi_pay["program_year"], errors="coerce")
    # Keep only payments within ±2 years of trial completion
    in_window = (
        df_inv_npi_pay["program_year"].notna()
        & df_inv_npi_pay["completion_year"].notna()
        & (abs(df_inv_npi_pay["program_year"] - df_inv_npi_pay["completion_year"]) <= 2)
    )
    df_inv_windowed = df_inv_npi_pay[in_window].copy()
else:
    df_inv_windowed = df_inv_npi_ct.copy()
    df_inv_windowed["year_payment"] = np.nan

# Step 3: aggregate up to trial level
trial_pay_agg = (
    df_inv_npi_ct.groupby("nct_id", as_index=False)
    .agg(
        n_investigators           =("investigator_name", "nunique"),
        n_investigators_npi_found =("npi",               lambda x: x.notna().sum()),
        n_investigators_unique    =("match_type",         lambda x: (x == "exact_unique").sum()),
    )
)

# Sum windowed payments per trial
windowed_pay = (
    df_inv_windowed.groupby("nct_id", as_index=False)["year_payment"]
    .sum()
    .rename(columns={"year_payment": "trial_total_payment"})
)
windowed_count = (
    df_inv_windowed.groupby("nct_id", as_index=False)["year_payment"]
    .count()
    .rename(columns={"year_payment": "trial_payment_count"})
)
trial_pay_agg = trial_pay_agg.merge(windowed_pay,   on="nct_id", how="left")
trial_pay_agg = trial_pay_agg.merge(windowed_count, on="nct_id", how="left")

# Step 4: aggregate publications up to trial level
df_pub_agg = (
    df_pubs.groupby("nct_id", as_index=False)
    .agg(
        pub_count      =("pmid",     "count"),
        first_pub_date =("pub_date", "min"),
    )
)
df_pub_agg["has_publication"] = 1

# Step 5: merge everything onto the main trials table
df_master = (
    df_trials
    .merge(trial_pay_agg, on="nct_id", how="left")
    .merge(df_pub_agg,    on="nct_id", how="left")
)

df_master["has_publication"] = df_master["has_publication"].fillna(0).astype(int)
df_master["pub_count"]       = df_master["pub_count"].fillna(0).astype(int)

# Timing columns
df_master["days_to_publication"] = (
    (df_master["first_pub_date"] - df_master["completion_date"]).dt.days
)

# Continuous payment predictor for modeling: log(1 + total windowed payment)
df_master["log1p_payment"] = np.log1p(df_master["trial_total_payment"].fillna(0))

# Payment tier (quartile among trials with positive windowed payment)
has_pay = df_master["trial_total_payment"].notna() & (df_master["trial_total_payment"] > 0)
df_master["payment_tier"] = "no_payment_data"
if has_pay.sum() >= 4:
    df_master.loc[has_pay, "payment_tier"] = pd.qcut(
        df_master.loc[has_pay, "trial_total_payment"],
        q=4,
        labels=["Q1_low", "Q2", "Q3", "Q4_high"],
        duplicates="drop",
    ).astype(str)

# Trial-level linkage confidence stratum
df_master["linkage_confidence"] = np.select(
    [
        df_master["n_investigators_unique"] == df_master["n_investigators"],
        df_master["n_investigators_npi_found"] > 0,
    ],
    ["all_exact", "partial"],
    default="none",
)

df_master.to_csv(MOD / "trial_payment_publication_master.csv", index=False)

print("Master table shape       :", df_master.shape)
print("Has results              :", df_master["has_results"].sum(), "/", len(df_master))
print("Results posted <= 2yr    :", df_master["results_posted_2yr"].sum(), "/", len(df_master))
print("Has publication          :", df_master["has_publication"].sum(), "/", len(df_master))
print("Has windowed payment data:", has_pay.sum(), "/", len(df_master))
print("\nLinkage confidence strata:")
print(df_master["linkage_confidence"].value_counts())
print("\nPayment tier distribution:")
print(df_master["payment_tier"].value_counts())
df_master[[
    "nct_id", "results_posted_2yr", "days_to_results", "has_publication",
    "trial_total_payment", "log1p_payment", "payment_tier",
    "log_enrollment", "condition_group", "linkage_confidence",
]].head(10)

---
## 6) EDA

Midterm EDA deliverables (per Option D spec):
- **Payment distributions** by payment nature and top companies
- **Results-posting and publication rates** by sponsor class and payment tier
- **Linkage QA**: match-confidence strata visualization and manual sample table
- **Association exploration**: payment level vs results/publication outcomes

In [ ]:
# --- Plot 1: Top 15 companies by total payment amount ---
if "company_name" in df_payments.columns and "payment_amount" in df_payments.columns and len(df_payments) > 0:
    top_co = (
        df_payments.groupby("company_name", as_index=False)["payment_amount"]
        .sum()
        .nlargest(15, "payment_amount")
        .sort_values("payment_amount")
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_co["company_name"], top_co["payment_amount"] / 1e6, color="#3b82f6")
    ax.set_xlabel("Total Payments (USD millions)")
    ax.set_title("Top 15 Companies by Total General Payments\n(investigators from completed oncology trials, 2015–2022)")
    fig.tight_layout()
    fig.savefig(RESULTS / "top_companies_by_payment.png", dpi=200)
    plt.show()
    print("Saved: top_companies_by_payment.png")
    top_co.to_csv(MOD / "op_d_top_companies.csv", index=False)
else:
    print("Skipping company plot: insufficient payment data.")

# --- Plot 2: Payment nature — count and total amount ---
if "payment_nature" in df_payments.columns and len(df_payments) > 0:
    nat_counts = (
        df_payments.groupby("payment_nature", as_index=False)
        .agg(count=("payment_nature", "size"), total=("payment_amount", "sum"))
        .sort_values("total", ascending=False)
        .head(12)
        .sort_values("total")
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(nat_counts["payment_nature"], nat_counts["count"],        color="#10b981")
    axes[1].barh(nat_counts["payment_nature"], nat_counts["total"] / 1e6, color="#f59e0b")
    axes[0].set_xlabel("Number of payment records")
    axes[1].set_xlabel("Total USD (millions)")
    axes[0].set_title("Payment count by nature (top 12)")
    axes[1].set_title("Total amount by payment nature (top 12)")
    fig.tight_layout()
    fig.savefig(RESULTS / "payment_dist_by_nature.png", dpi=200)
    plt.show()
    print("Saved: payment_dist_by_nature.png")
    nat_counts.to_csv(MOD / "op_d_payment_nature_summary.csv", index=False)
else:
    print("Skipping nature plot: insufficient payment data.")

# --- Plot 3: log10(payment+1) distribution by sponsor type ---
pay_df = df_master[df_master["trial_total_payment"].notna()].copy()
pay_df["log1p_pay"] = np.log10(pay_df["trial_total_payment"].clip(lower=0) + 1)
if len(pay_df) > 5:
    fig, ax = plt.subplots(figsize=(9, 5))
    for stype, grp in pay_df.groupby("lead_sponsor_class"):
        ax.hist(grp["log1p_pay"], bins=30, alpha=0.6, label=stype, density=True)
    ax.set_xlabel("log₁₀(Total investigator payments + 1, USD)")
    ax.set_ylabel("Density")
    ax.set_title("Payment Distribution by Lead Sponsor Class")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS / "payment_dist_by_sponsor_type.png", dpi=200)
    plt.show()
    print("Saved: payment_dist_by_sponsor_type.png")
else:
    print("Skipping sponsor distribution plot: insufficient data.")

In [ ]:
df_eda = df_master.copy()

# --- Plot 4: Results-posting (2yr) and overall posting rate by sponsor class ---
sponsor_outcomes = (
    df_eda.groupby("lead_sponsor_class", as_index=False)
    .agg(
        n_trials        =("nct_id",             "count"),
        n_results       =("has_results",         "sum"),
        n_results_2yr   =("results_posted_2yr",  "sum"),
        n_publications  =("has_publication",     "sum"),
    )
)
sponsor_outcomes["results_rate"]     = sponsor_outcomes["n_results"]      / sponsor_outcomes["n_trials"] * 100
sponsor_outcomes["results_2yr_rate"] = sponsor_outcomes["n_results_2yr"]  / sponsor_outcomes["n_trials"] * 100
sponsor_outcomes["pub_rate"]         = sponsor_outcomes["n_publications"]  / sponsor_outcomes["n_trials"] * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=sponsor_outcomes, x="lead_sponsor_class", y="results_rate",     ax=axes[0], color="#3b82f6")
sns.barplot(data=sponsor_outcomes, x="lead_sponsor_class", y="results_2yr_rate", ax=axes[1], color="#6366f1")
sns.barplot(data=sponsor_outcomes, x="lead_sponsor_class", y="pub_rate",         ax=axes[2], color="#10b981")
axes[0].set_title("Results-Posting Rate by Sponsor Class")
axes[1].set_title("Results Posted ≤2yr Rate by Sponsor Class")
axes[2].set_title("Publication Rate by Sponsor Class")
for ax in axes:
    ax.set_ylabel("% of trials")
    ax.set_xlabel("Sponsor class")
fig.tight_layout()
fig.savefig(RESULTS / "results_pub_rate_by_sponsor.png", dpi=200)
plt.show()
print("Saved: results_pub_rate_by_sponsor.png")
sponsor_outcomes.to_csv(MOD / "trial_outcomes_by_sponsor.csv", index=False)

# --- Plot 5: Results-posting (2yr) rate by payment tier ---
tier_outcomes = (
    df_eda.groupby("payment_tier", as_index=False)
    .agg(
        n_trials       =("nct_id",            "count"),
        n_results      =("has_results",        "sum"),
        n_results_2yr  =("results_posted_2yr", "sum"),
        n_publications =("has_publication",    "sum"),
    )
)
tier_outcomes["results_rate"]     = tier_outcomes["n_results"]      / tier_outcomes["n_trials"] * 100
tier_outcomes["results_2yr_rate"] = tier_outcomes["n_results_2yr"]  / tier_outcomes["n_trials"] * 100
tier_outcomes["pub_rate"]         = tier_outcomes["n_publications"]  / tier_outcomes["n_trials"] * 100

tier_order = ["no_payment_data", "Q1_low", "Q2", "Q3", "Q4_high"]
tier_outcomes["payment_tier"] = pd.Categorical(
    tier_outcomes["payment_tier"], categories=tier_order, ordered=True
)
tier_outcomes = tier_outcomes.sort_values("payment_tier")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=tier_outcomes, x="payment_tier", y="results_2yr_rate", ax=axes[0], color="#6366f1")
sns.barplot(data=tier_outcomes, x="payment_tier", y="pub_rate",         ax=axes[1], color="#f43f5e")
axes[0].set_title("Results Posted ≤2yr Rate by Payment Tier")
axes[1].set_title("Publication Rate by Payment Tier")
axes[0].set_ylabel("% of trials with results posted within 2yr")
axes[1].set_ylabel("% of trials with linked publication")
for ax in axes:
    ax.set_xlabel("Payment tier (windowed investigator total)")
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(RESULTS / "results_pub_rate_by_payment_tier.png", dpi=200)
plt.show()
print("Saved: results_pub_rate_by_payment_tier.png")
tier_outcomes.to_csv(MOD / "trial_outcomes_by_payment_tier.csv", index=False)

# --- Plot 6: Temporal trend — results-posted-2yr rate by completion year ---
year_outcomes = (
    df_eda[df_eda["completion_year"].between(2015, 2022)]
    .groupby("completion_year", as_index=False)
    .agg(
        n_trials      =("nct_id",            "count"),
        n_results_2yr =("results_posted_2yr", "sum"),
    )
)
year_outcomes["results_2yr_rate"] = year_outcomes["n_results_2yr"] / year_outcomes["n_trials"] * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(year_outcomes["completion_year"], year_outcomes["n_trials"], color="#94a3b8", alpha=0.5, label="Trial count")
ax2.plot(year_outcomes["completion_year"], year_outcomes["results_2yr_rate"], "o-", color="#6366f1", label="% results posted ≤2yr")
ax1.set_xlabel("Primary completion year")
ax1.set_ylabel("Number of completed oncology trials")
ax2.set_ylabel("% with results posted ≤2yr")
ax1.set_title("Completed Oncology Trials and Timely Results Posting Rate (2015–2022)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
fig.tight_layout()
fig.savefig(RESULTS / "temporal_trend_results_posting.png", dpi=200)
plt.show()
print("Saved: temporal_trend_results_posting.png")
year_outcomes.to_csv(MOD / "trial_outcomes_by_year.csv", index=False)

In [ ]:
# --- Plot 5: Investigator and trial-level linkage confidence strata ---
strata_inv  = df_npi["match_type"].value_counts().reset_index()
strata_inv.columns = ["match_type", "count"]

conf_counts = df_master["linkage_confidence"].value_counts().reset_index()
conf_counts.columns = ["confidence", "count"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=strata_inv,  x="match_type",  y="count", ax=axes[0], color="#8b5cf6")
sns.barplot(data=conf_counts, x="confidence",  y="count", ax=axes[1], color="#0ea5e9")
axes[0].set_title("Investigator NPI Match Strata\n(NPI Registry lookup by name)")
axes[1].set_title("Trial-Level Linkage Confidence\n(based on investigator NPI match quality)")
axes[0].set_xlabel("Match type")
axes[1].set_xlabel("Confidence stratum")
axes[0].set_ylabel("Number of investigators")
axes[1].set_ylabel("Number of trials")
fig.tight_layout()
fig.savefig(RESULTS / "linkage_qa_match_strata.png", dpi=200)
plt.show()
print("Saved: linkage_qa_match_strata.png")

# Manual sample verification: 10 exact-unique matches for QA review
exact_inv = df_inv_npi[df_inv_npi["match_type"] == "exact_unique"]
n_sample  = min(10, len(exact_inv))
if n_sample > 0:
    sample_qa = (
        exact_inv[["nct_id", "investigator_name", "role", "affiliation", "npi"]]
        .drop_duplicates(subset=["investigator_name"])
        .sample(n_sample, random_state=42)
        .reset_index(drop=True)
    )
    sample_qa.to_csv(MOD / "linkage_qa_sample_exact_unique.csv", index=False)
    print("\nSample exact-unique matches (manual QA):")
    print(sample_qa.to_string(index=False))
else:
    print("No exact-unique matches available for QA sample.")

In [ ]:
# --- Plot 7: Payment total vs results-posting status (box + histogram) ---
pay_df = df_master[
    df_master["trial_total_payment"].notna() & (df_master["trial_total_payment"] > 0)
].copy()
pay_df["log_payment"] = np.log10(pay_df["trial_total_payment"].clip(lower=1))
pay_df["results_2yr_label"] = pay_df["results_posted_2yr"].map({1: "Posted ≤2yr", 0: "Not posted ≤2yr"})

if len(pay_df) > 5:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.histplot(pay_df["log_payment"], bins=30, ax=axes[0], color="#f59e0b")
    axes[0].set_xlabel("log₁₀(Total investigator payments, USD)")
    axes[0].set_title("Distribution of Trial-Level Windowed Payment Totals")

    sns.boxplot(
        data=pay_df, x="results_2yr_label", y="log_payment", ax=axes[1],
        palette={"Posted ≤2yr": "#10b981", "Not posted ≤2yr": "#f43f5e"},
    )
    axes[1].set_xlabel("")
    axes[1].set_ylabel("log₁₀(Total payments, USD)")
    axes[1].set_title("Payment Total by 2-Year Results-Posting Status")

    fig.tight_layout()
    fig.savefig(RESULTS / "payment_dist_vs_results.png", dpi=200)
    plt.show()
    print("Saved: payment_dist_vs_results.png")
else:
    print("Insufficient payment data for correlation plots — skipping.")

# --- Plot 8: Payment box plot by trial phase ---
if len(pay_df) > 5 and "phase" in pay_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    phase_order = sorted(pay_df["phase"].dropna().unique())
    sns.boxplot(data=pay_df, x="phase", y="log_payment", order=phase_order, ax=ax, color="#818cf8")
    ax.set_xlabel("Trial phase")
    ax.set_ylabel("log₁₀(Total payments, USD)")
    ax.set_title("Payment Distribution by Trial Phase")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(RESULTS / "payment_dist_by_phase.png", dpi=200)
    plt.show()
    print("Saved: payment_dist_by_phase.png")

# --- Plot 9: Confounding audit — payment vs log(enrollment) ---
enroll_df = df_master[df_master["enrollment"].notna() & df_master["trial_total_payment"].notna()].copy()
enroll_df["log_pay"] = np.log1p(enroll_df["trial_total_payment"])
if len(enroll_df) > 10:
    from scipy import stats as _stats
    rho, pval = _stats.spearmanr(enroll_df["log_pay"], enroll_df["log_enrollment"])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(enroll_df["log_enrollment"], enroll_df["log_pay"], alpha=0.3, s=15, color="#6366f1")
    ax.set_xlabel("log(enrollment + 1)")
    ax.set_ylabel("log(1 + total payment, USD)")
    ax.set_title(f"Spearman ρ = {rho:.3f}  (p = {pval:.3e})")
    fig.tight_layout()
    fig.savefig(RESULTS / "confound_payment_vs_enrollment.png", dpi=200)
    plt.show()
    print(f"Saved: confound_payment_vs_enrollment.png  (ρ={rho:.3f}, p={pval:.3e})")
else:
    print("Insufficient linked data for enrollment–payment scatter — skipping.")

# Descriptive summary table for modeling
model_cols = [
    "nct_id", "lead_sponsor_class", "is_industry", "phase",
    "enrollment", "log_enrollment",
    "condition_group", "completion_year",
    "has_results", "results_posted_2yr", "days_to_results",
    "has_publication",
    "trial_total_payment", "log1p_payment",
    "payment_tier", "linkage_confidence",
    "days_to_publication",
]
df_analysis = df_master[[c for c in model_cols if c in df_master.columns]].copy()
df_analysis.to_csv(MOD / "trial_analysis_dataset.csv", index=False)
print(f"\nFinal analysis dataset saved: {len(df_analysis)} rows × {df_analysis.shape[1]} cols")
print("Columns:", list(df_analysis.columns))
df_analysis.describe(include="all").T[["count", "unique", "mean", "std"]].dropna(how="all").head(20)

In [ ]:
summary_paths = [
    # Raw
    RAW / "ct_d_version.json",
    *sorted(RAW.glob("ct_d_studies_page*.json")),
    RAW / "op_catalog.json",
    RAW / "op_general_payments_matched.json",
    # Modified — source tables
    MOD / "ct_d_trials_flat.csv",
    MOD / "ct_d_investigators.csv",
    MOD / "ct_d_investigator_npi_lookup.csv",
    MOD / "op_d_payments_flat.csv",
    MOD / "op_d_payments_by_npi_year.csv",
    MOD / "op_d_payments_by_npi.csv",
    MOD / "op_d_top_companies.csv",
    MOD / "op_d_payment_nature_summary.csv",
    MOD / "pubmed_d_publications.csv",
    # Modified — linkage & outcomes
    MOD / "trial_payment_publication_master.csv",
    MOD / "trial_outcomes_by_sponsor.csv",
    MOD / "trial_outcomes_by_payment_tier.csv",
    MOD / "trial_outcomes_by_year.csv",
    MOD / "linkage_qa_sample_exact_unique.csv",
    # Final modeling dataset
    MOD / "trial_analysis_dataset.csv",
    # Results / figures
    RESULTS / "top_companies_by_payment.png",
    RESULTS / "payment_dist_by_nature.png",
    RESULTS / "payment_dist_by_sponsor_type.png",
    RESULTS / "results_pub_rate_by_sponsor.png",
    RESULTS / "results_pub_rate_by_payment_tier.png",
    RESULTS / "temporal_trend_results_posting.png",
    RESULTS / "payment_dist_vs_results.png",
    RESULTS / "payment_dist_by_phase.png",
    RESULTS / "confound_payment_vs_enrollment.png",
    RESULTS / "linkage_qa_match_strata.png",
]

print("Output summary:")
for p in summary_paths:
    status = "OK  " if p.exists() else "MISS"
    print(f"  [{status}] {p}")

# Print key modeling dataset column list
if (MOD / "trial_analysis_dataset.csv").exists():
    import pandas as _pd
    _df = _pd.read_csv(MOD / "trial_analysis_dataset.csv", nrows=0)
    print(f"\nFinal analysis dataset columns ({len(_df.columns)}):")
    print(list(_df.columns))